# Fase 3.2 — Optimización y diagnóstico con Optuna

Este notebook **reemplaza el rol de `src/optuna_optimization.py`**: busca los mejores hiperparámetros de **XGBoost, Random Forest, LightGBM y Ridge** y permite analizar visualmente los experimentos.

**Flujo:** cargar Train/Validation del preprocesamiento → ejecutar trials → registrar métricas → exportar `models/best_params.json` → analizar los límites de búsqueda con gráficos de **coordenadas paralelas (los "hilos")**.

> **No se usa Test para entrenar, optimizar ni seleccionar.** Las métricas de aquí son de validación. Si cambia el preprocesamiento, hay que volver a ejecutar la optimización. No hace falta modificar `train_model.py` para ejecutar este notebook.

## 0. Dependencias y conexión con el proyecto

Ejecutá el notebook desde el repositorio (`notebooks/optuna_optimization.ipynb`) o desde la raíz del proyecto. Necesitás `optuna`, `plotly`, `pandas`, `scikit-learn`, `xgboost` y `lightgbm` en **el mismo intérprete de Jupyter**. Si falta algo: `python -m pip install optuna plotly` (o con `uv pip install optuna plotly`).

Reutilizamos `cargar_datos_preprocesados`, `crear_modelo` y `registrar_experimento` de tu `src/train_model.py`: así no duplicamos lógica de entrenamiento.

In [ ]:
from pathlib import Path
import hashlib
import json
import sys
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import optuna
from IPython.display import display

# Encontrar la raíz tanto si Jupyter se abrió desde notebooks/ como desde la raíz.
CANDIDATOS = [Path.cwd(), Path.cwd().parent]
RAIZ = next((p for p in CANDIDATOS if (p / "src" / "train_model.py").is_file()), None)
if RAIZ is None:
    raise FileNotFoundError("No encuentro src/train_model.py. Abrí Jupyter desde la raíz o notebooks/ del repositorio.")

sys.path.insert(0, str(RAIZ / "src"))
from train_model import (  # noqa: E402
    cargar_datos_preprocesados,
    crear_modelo,
    registrar_experimento,
)

print("Raíz del proyecto:", RAIZ)
print("Optuna:", optuna.__version__)

## 1. Configuración

- `N_TRIALS`: pruebas por algoritmo. **20** es un primer ensayo, no un estudio exhaustivo. Para inspeccionar bien los gráficos, conviene aumentar a 50–100 cuando el preprocesamiento sea confiable.
- `MODELOS`: elegí cuáles optimizar; por defecto los cuatro.
- `RUTA_CSV`: `None` usa la ruta configurada en `train_model.py`.

**Los rangos de búsqueda son los mismos que el `.py` original.** Más abajo veremos si conviene ampliarlos.

In [ ]:
N_TRIALS = 20
MODELOS = ["XGBoost", "Random Forest", "LightGBM", "Ridge"]
RUTA_CSV = None  # o, por ejemplo: RAIZ / "data" / "processed" / "dataset_final.csv"
SEMILLA = 42

if N_TRIALS < 1:
    raise ValueError("N_TRIALS debe ser al menos 1")

## 2. Cargar los datos de preprocesamiento

Usamos **Train** para ajustar y **Validation** para obtener el MAE que guía a Optuna. El conjunto **Test permanece sin usar**.

In [ ]:
X_train, y_train, X_val, y_val, _X_test, _y_test = cargar_datos_preprocesados(RUTA_CSV)
print(f"Train: {X_train.shape} | Validation: {X_val.shape} | Test: reservado")

## 3. Espacio de búsqueda de hiperparámetros

Esta función usa las mismas configuraciones que veníamos probando con Optuna. Los rangos quedan explícitos para documentarlos y compararlos con el diagnóstico final.

In [ ]:
def sugerir_parametros(trial, nombre):
    if nombre == "XGBoost":
        return {
            "n_estimators": trial.suggest_int("n_estimators", 100, 500, step=100),
            "max_depth": trial.suggest_int("max_depth", 2, 10),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        }
    if nombre == "Random Forest":
        return {
            "n_estimators": trial.suggest_int("n_estimators", 100, 400, step=50),
            "max_depth": trial.suggest_int("max_depth", 3, 20),
            "min_samples_split": trial.suggest_int("min_samples_split", 2, 10),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 5),
        }
    if nombre == "LightGBM":
        return {
            "n_estimators": trial.suggest_int("n_estimators", 100, 500, step=100),
            "num_leaves": trial.suggest_int("num_leaves", 15, 63),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
            "max_depth": trial.suggest_int("max_depth", 3, 12),
        }
    if nombre == "Ridge":
        return {"alpha": trial.suggest_float("alpha", 0.001, 1000, log=True)}
    raise ValueError(f"Modelo desconocido: {nombre}")

## 4. Optimizar y registrar cada experimento

Cada trial ajusta el modelo con **Train**, mide MAE/RMSE/R² en **Validation** y llama a `registrar_experimento()` (tu función actual) para conservar el historial en `results/experiment_logs.csv`.

**No ejecutamos `train_model.py` como script**, porque eso repetiría la optimización. Solo importamos sus funciones.

`estudios` conserva los objetos de Optuna necesarios para los gráficos.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

estudios = {}
resultados = {}
mejores_parametros = {}

for nombre in MODELOS:
    print(f"\n===== {nombre}: {N_TRIALS} trials =====")

    def objetivo(trial):
        params = sugerir_parametros(trial, nombre)
        modelo = crear_modelo(nombre, params)
        modelo.fit(X_train, y_train)
        pred = modelo.predict(X_val)
        metricas = {
            "MAE": float(mean_absolute_error(y_val, pred)),
            "RMSE": float(mean_squared_error(y_val, pred) ** 0.5),
            "R2": float(r2_score(y_val, pred)),
        }
        trial.set_user_attr("RMSE", metricas["RMSE"])
        trial.set_user_attr("R2", metricas["R2"])
        registrar_experimento(f"{nombre} - trial {trial.number}", modelo, metricas)
        return metricas["MAE"]

    estudio = optuna.create_study(
        direction="minimize",
        study_name=nombre,
        sampler=optuna.samplers.TPESampler(seed=SEMILLA),
    )
    estudio.optimize(objetivo, n_trials=N_TRIALS, n_jobs=1)

    estudios[nombre] = estudio
    mejores_parametros[nombre] = estudio.best_params
    resultados[nombre] = {
        "MAE": estudio.best_value,
        "RMSE": estudio.best_trial.user_attrs["RMSE"],
        "R2": estudio.best_trial.user_attrs["R2"],
        "trial": estudio.best_trial.number,
    }
    print(f"Mejor MAE ({nombre}): {estudio.best_value:,.2f}")
    print("Parámetros:", estudio.best_params)

tabla = pd.DataFrame.from_dict(resultados, orient="index").sort_values("MAE")
print("\nResultados en VALIDACIÓN (NO en test):")
display(tabla)

## 5. Guardar hiperparámetros y resultados

Se genera **`models/best_params.json`** con el formato que `train_model.py` puede leer más adelante (un diccionario por modelo). También guardamos el detalle de cada trial como CSV, para analizarlo sin perder los resultados.

Además dejamos un archivo de metadatos con la fecha y la huella SHA-256 del CSV procesado para recordar **sobre qué versión de datos** se hizo la búsqueda.

**Si solo optimizás algunos modelos, no borramos los parámetros guardados de los demás.**

In [ ]:
carpeta_models = RAIZ / "models"
carpeta_results = RAIZ / "results"
carpeta_models.mkdir(parents=True, exist_ok=True)
carpeta_results.mkdir(parents=True, exist_ok=True)

ruta_params = carpeta_models / "best_params.json"
params_guardados = {}
if ruta_params.exists():
    with ruta_params.open(encoding="utf-8") as f:
        params_guardados = json.load(f)
params_guardados.update(mejores_parametros)
with ruta_params.open("w", encoding="utf-8") as f:
    json.dump(params_guardados, f, ensure_ascii=False, indent=2)

for nombre, estudio in estudios.items():
    nombre_archivo = nombre.lower().replace(" ", "_")
    estudio.trials_dataframe().to_csv(
        carpeta_results / f"optuna_trials_{nombre_archivo}.csv", index=False
    )

# Encontrar exactamente el archivo utilizado por la función de carga.
if RUTA_CSV is not None:
    ruta_datos = Path(RUTA_CSV).expanduser().resolve()
else:
    opciones = [RAIZ / "data" / "processed" / "dataset_final.csv",
                RAIZ / "notebooks" / "data" / "processed" / "dataset_final.csv"]
    ruta_datos = next((p for p in opciones if p.is_file()), None)

huella = None
if ruta_datos is not None:
    sha = hashlib.sha256()
    with ruta_datos.open("rb") as f:
        for bloque in iter(lambda: f.read(1024 * 1024), b""):
            sha.update(bloque)
    huella = sha.hexdigest()

metadata = {
    "fecha_utc": datetime.now(timezone.utc).isoformat(),
    "dataset": str(ruta_datos) if ruta_datos else None,
    "dataset_sha256": huella,
    "modelos_ejecutados": MODELOS,
    "n_trials_por_modelo": N_TRIALS,
    "semilla": SEMILLA,
    "metrica_optimizacion": "MAE_validacion",
}
with (carpeta_results / "optuna_run_metadata.json").open("w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print(f"Hiperparámetros: {ruta_params}")
print(f"Trials detallados: {carpeta_results}")

## 6. Diagnóstico visual de Optuna

Elegí un modelo en `MODELO_DIAGNOSTICO`. Los gráficos son **interactivos** (Plotly).

1. **Coordenadas paralelas («hilos»):** cada línea es un trial y cada eje representa un hiperparámetro. El color refleja el MAE. Sirve para ver qué combinaciones funcionan mejor y si los buenos resultados se amontonan cerca de un límite.
2. **Historial de optimización:** muestra si el mejor MAE sigue mejorando a medida que probamos nuevas configuraciones.
3. **Cortes por parámetro (`slice`):** MAE frente a cada parámetro para descubrir tendencias.
4. **Importancia de hiperparámetros:** indica cuáles explican más variación del MAE *dentro del espacio explorado*.

Con pocos trials, la importancia y los patrones visuales son orientativos.

In [ ]:
import optuna.visualization as ov

MODELO_DIAGNOSTICO = "XGBoost"  # Cambiar por: "Random Forest", "LightGBM" o "Ridge"
if MODELO_DIAGNOSTICO not in estudios:
    raise ValueError(f"No se optimizó {MODELO_DIAGNOSTICO}. Disponibles: {list(estudios)}")

estudio = estudios[MODELO_DIAGNOSTICO]
print(f"Diagnóstico de {MODELO_DIAGNOSTICO}: {len(estudio.trials)} trials")

print("1) Coordenadas paralelas (hilos)")
ov.plot_parallel_coordinate(estudio, target_name="MAE de validación").show()

print("2) Historial de optimización")
ov.plot_optimization_history(estudio, target_name="MAE de validación").show()

print("3) MAE vs cada hiperparámetro")
ov.plot_slice(estudio, target_name="MAE de validación").show()

if len(estudio.trials) >= 10 and len(estudio.best_params) > 1:
    print("4) Importancia de hiperparámetros")
    try:
        ov.plot_param_importances(estudio, target_name="MAE de validación").show()
    except ValueError as exc:
        print("No se pudo estimar la importancia con estos trials:", exc)
else:
    print("4) Para importancia de parámetros, probá al menos 10 trials y más de un parámetro.")

## 7. ¿Los rangos de búsqueda están bien acotados?

El siguiente diagnóstico revisa **dónde cayó el mejor trial** respecto de los límites definidos. Una solución cerca de un extremo puede sugerir ampliar el rango, **pero no lo demuestra**: hace falta observar varios trials buenos y repetir la búsqueda con más presupuesto.

Para parámetros con escala logarítmica (ej. `alpha`, `learning_rate`), la proximidad se mide en **escala logarítmica**, no lineal.

In [ ]:
from optuna.distributions import FloatDistribution, IntDistribution


def diagnosticar_limites(estudio, umbral=0.10):
    filas = []
    mejor = estudio.best_trial
    for nombre, distribucion in mejor.distributions.items():
        if not isinstance(distribucion, (FloatDistribution, IntDistribution)):
            continue
        valor = mejor.params[nombre]
        inferior, superior = distribucion.low, distribucion.high
        if inferior == superior:
            continue
        if getattr(distribucion, "log", False):
            posicion = (np.log(valor) - np.log(inferior)) / (np.log(superior) - np.log(inferior))
        else:
            posicion = (valor - inferior) / (superior - inferior)
        borde = ("cerca del MÍNIMO" if posicion <= umbral else
                 "cerca del MÁXIMO" if posicion >= 1 - umbral else
                 "interior")
        filas.append({
            "Hiperparámetro": nombre,
            "Mínimo": inferior,
            "Mejor valor": valor,
            "Máximo": superior,
            "Posición relativa": round(float(posicion), 3),
            "Diagnóstico": borde,
        })
    return pd.DataFrame(filas)

for nombre, estudio in estudios.items():
    print(f"\n=== {nombre} | mejor MAE: {estudio.best_value:,.2f} ===")
    display(diagnosticar_limites(estudio))